# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane: CTR / Engagement Opportunity Scoring.**

Task type: **scoring** (built on a regression model), used to produce a **ranked** list — not
plain classification and not clustering. The underlying decision is "which ones first?", which
the framing guide maps to ranking/scoring with a priority score as the target.

Concretely: I train a regression model to predict a page's **expected CTR** from its position,
intent, content type, and other observed features. The **opportunity score** is the gap between
a page's *actual* CTR and its *model-expected* CTR (`ctr_gap = ctr - predicted_ctr`). Pages with
the most negative gap — doing *worse* than pages like them, at the same position — are the ones
an editor should look at first. This is a scoring task with a regression model underneath it,
not a yes/no classification of "good" vs "bad" pages.


In [1]:
# No computation needed yet — this cell documents the decision made above.
lane = "CTR / Engagement Opportunity Scoring"
task_type = "scoring (regression-based ranking)"
print(f"Lane: {lane}")
print(f"ML task type: {task_type}")


Lane: CTR / Engagement Opportunity Scoring
ML task type: scoring (regression-based ranking)


## 2. Target or proxy

**Target for the regression model: `ctr` (click-through rate over the trailing 90 days).**

This is an **observed** outcome — it's measured directly from GSC clicks and impressions
(`ctr = clicks_90d / impressions_90d * 100`, roughly), not a label someone defined by rule. That
satisfies the "target must be observed, not defined" rule: I'm not asking the model to learn
someone's existing scoring formula, I'm asking it to learn the real relationship between page
features and real click behavior.

The **opportunity score** (`ctr_gap = ctr - predicted_ctr`) is a *derived proxy* built on top of
that observed target, used only for ranking after the model is trained — it is not itself the
training label. I'm keeping that distinction explicit because it's easy to blur "what I predict"
with "what I rank by."

One honest caveat up front: `avg_position = 0` means "no position data," not "rank zero" — those
1,205 rows get dropped before modeling, not treated as top-ranked pages.


In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# avg_position == 0 means "no data", not "rank zero" -- drop before anything else
df = df[df["avg_position"] > 0].copy()

print(f"Rows after dropping no-position rows: {len(df):,}")
print(df[["content_id", "avg_position", "ctr", "impressions_90d"]].head())


Rows after dropping no-position rows: 28,795
             content_id  avg_position   ctr  impressions_90d
0  content_304f48230142          10.6  0.76             3803
1  content_a1fb4e703a9e          20.3  0.05            15320
2  content_9aa793d4d895          36.5  0.09            12581
3  content_331d6c4de07b           6.2  0.49            11751
4  content_d99b7a2d90ca          44.0  0.13            19140


## 3. Success metric

**Metric: held-out MAE (mean absolute error) of predicted CTR vs. actual CTR, benchmarked
against a position-tier-average baseline.**

Concretely:
- Baseline: for each page, predict CTR as the median CTR of its `position_tier` bucket
  (`top_3`, `page_1`, `striking`, `page_3_5`, `deep`). This is the "fixed rule" — one lookup
  table, five numbers.
- Model: a regression using position, intent, content type, word count, and freshness, evaluated
  on a **client-holdout split** (some clients' pages never appear in training — same discipline
  the shipped pipeline uses for the refresh lane), scored by MAE on the held-out clients.
- "Good" means: model MAE is meaningfully lower than baseline MAE on the same held-out rows —
  not just a smaller decimal, a real, repeatable gap. I'm not defining "good" after seeing the
  number; the comparison is baseline-vs-model MAE, decided before training.

I chose MAE over R² because MAE is in the same units as CTR (percentage points) and is easy to
explain to a non-technical stakeholder: "the model's CTR guess is typically off by X points,
versus Y points for the position-only rule."

**Volume caveat:** low-impression pages have noisy CTR (one extra click swings the rate a lot),
so any real modeling pass needs a minimum-impressions filter before this metric is trustworthy —
noted here, applied when I get to the actual model-building week, not faked in this framing pass.

**Honest flag from checking the baseline (see code cell below):** with the volume filter applied,
`top_3` pages don't have the highest median CTR — `page_1` does, by a small margin, and each
tier has hundreds to thousands of rows behind it, so this isn't just small-sample noise. I'm not
explaining it away here; it just means the position-only rule is a weaker baseline than "better
position always wins," which is exactly the kind of thing a signal audit (later in the track)
needs to dig into before I trust either the rule or the model.


In [3]:
success_metric = "Held-out MAE (predicted CTR vs actual CTR), model vs position-tier-average baseline"
print(success_metric)

# Quick honesty check: does the position-tier baseline even separate CTR today,
# and is each tier's median backed by enough rows to trust it (not just 5 pages)?
MIN_IMPRESSIONS_CHECK = 100
checked = df[df["impressions_90d"] >= MIN_IMPRESSIONS_CHECK]
print(checked.groupby("position_tier")["ctr"].agg(["count", "median"]))


Held-out MAE (predicted CTR vs actual CTR), model vs position-tier-average baseline
               count  median
position_tier               
deep             879    0.00
page_1          8633    0.23
page_3_5        6058    0.06
striking        5903    0.15
top_3            533    0.19


## 4. The unit of analysis, as a real dataframe

**One row = one (client, content item) pair, summarized over its trailing 90 days.**

That's the grain the starter CSV ships at, and it's the right grain for this lane: I'm scoring
individual pages, not clients and not queries. Below is the actual slice of columns this lane
needs, with a minimum-impressions filter applied so the CTR numbers aren't just noise from
pages with a handful of impressions.


In [4]:
MIN_IMPRESSIONS = 100  # drop pages too sparse for a stable CTR estimate

lane_cols = [
    "content_id", "client_id",
    "avg_position", "position_tier",
    "impressions_90d", "impression_tier",
    "ctr", "engagement_rate", "scroll_rate",
    "content_type", "main_intent", "word_count", "freshness_tier",
]

lane_df = df.loc[df["impressions_90d"] >= MIN_IMPRESSIONS, lane_cols].reset_index(drop=True)

print(f"Unit of analysis: one row = one (client_id, content_id) page, trailing-90-day window")
print(f"Rows after >= {MIN_IMPRESSIONS} impressions filter: {len(lane_df):,} of {len(df):,}")
lane_df.head()


Unit of analysis: one row = one (client_id, content_id) page, trailing-90-day window
Rows after >= 100 impressions filter: 22,006 of 28,795


,content_id,client_id,avg_position,position_tier,impressions_90d,impression_tier,ctr,engagement_rate,scroll_rate,content_type,main_intent,word_count,freshness_tier
0,content_304f48230142,client_f369cb89fc,10.6,striking,3803,good,0.76,5.88,4.55,keyword article,transactional,3221.0,0-30
1,content_a1fb4e703a9e,client_4e07408562,20.3,page_3_5,15320,good,0.05,0.00,10.00,keyword article,informational,2481.0,0-30
2,content_9aa793d4d895,client_7f2253d7e2,36.5,page_3_5,12581,good,0.09,0.00,28.57,keyword article,informational,3515.0,0-30
3,content_331d6c4de07b,client_19581e27de,6.2,page_1,11751,good,0.49,1.28,3.45,keyword article,commercial,NaN,0-30
4,content_d99b7a2d90ca,client_3fdba35f04,44.0,page_3_5,19140,good,0.13,0.00,24.29,keyword article,informational,2803.0,0-30


## 5. Why ML beats a fixed rule here

A fixed rule already exists implicitly in `position_tier` — "pages ranking `top_3` should get
roughly X% CTR, `page_1` roughly Y%," and so on. That's five numbers in a lookup table, and it
would be the natural first thing to try (I even use it as the baseline above, on purpose).

But CTR at a given position isn't determined by position alone — it interacts with several other
signals at once:

- **Intent** (`main_intent`): a transactional query at position 5 gets clicked very differently
  than an informational one at the same position, because SERP features (shopping results,
  featured snippets) compete differently by intent.
- **Content type** and **word_count**: how a result *looks* in the SERP (title/meta patterns tied
  to content type and length) shifts CTR independent of rank.
- **Freshness** (`freshness_tier`): stale pages sitting at a good position may still lose clicks
  to fresher competing results the position number alone doesn't capture.

These factors don't just add up — they interact (e.g., freshness matters more for some intents
than others). A hand-written if/else rule can encode one or two of these at a time before it
becomes unreadable; a model can weigh all of them together and adjust as the data shifts. That's
the actual case for ML here: not that a rule is *impossible*, but that a rule capturing all these
interactions by hand would be an unmaintainable pile of thresholds, while a regression model
learns the combined pattern directly from the observed data — and I can *check* whether it
actually beats the simple rule (Section 3) instead of assuming it does.


In [5]:
# Sanity check that the interaction claim above is real, not asserted:
# does CTR vary by intent WITHIN the same position tier? If not, the "interaction" argument is weak.
check = (
    lane_df.groupby(["position_tier", "main_intent"])["ctr"]
    .median()
    .unstack()
)
check


main_intent,commercial,informational,navigational,transactional
position_tier,,,,
deep,0.00,0.00,0.095,0.00
page_1,0.22,0.22,0.325,0.25
page_3_5,0.06,0.06,0.100,0.07
striking,0.15,0.15,0.460,0.17
top_3,0.19,0.14,NaN,0.29


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.